In [25]:
import numpy as np
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"


# 1. Read data

In [26]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/per_length_per_kfold_split/' # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu
TARGET_PATH_TO_RESULTS = "experiment_results/alleles_with_groups_balanced/" #!CHANGE for bwv!

In [27]:
from data_reading_methods import get_available_alleles, read_data

ALLELES = get_available_alleles(DATA_PATH)

per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")

ALLELES

['HLA-A*01:01', 'HLA-B*27:05', 'HLA-A*02:01', 'HLA-A*03:01', 'HLA-B*07:02']

## 2.0 Merge alleles together

In [28]:
# combination_size = len(ALLELES)
ALLELES = ['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01']
combination_size = 3
TOTAL_SPLITS = 5

In [29]:
import itertools
from collections import defaultdict
per_allele_per_kfold_per_length_binders_train_new = dict()

per_allele_per_kfold_per_length_binders_test_new = dict()

per_kfold_per_length_per_allele_counts_old_train = dict()
per_kfold_per_length_per_allele_counts_old_test = dict()



NEW_ALELLES = list()
selected_lengths = np.arange(8,20)



for combination in itertools.combinations(ALLELES, combination_size):
    allele_name = "+".join(combination)
    NEW_ALELLES.append(allele_name)
    print(allele_name)
    per_allele_per_kfold_per_length_binders_train_new[allele_name] = dict()
    per_allele_per_kfold_per_length_binders_test_new[allele_name] = dict()


    for split_num in range(TOTAL_SPLITS):
        per_allele_per_kfold_per_length_binders_train_new[allele_name][split_num] = dict()
        per_allele_per_kfold_per_length_binders_test_new[allele_name][split_num] = dict()
        per_kfold_per_length_per_allele_counts_old_train[split_num] = dict()
        per_kfold_per_length_per_allele_counts_old_test[split_num] = dict()
        list_of_length_sets = list()
        for old_allele in combination:
            lengths = set(per_allele_per_kfold_per_length_binders_train[old_allele][split_num].keys())
            list_of_length_sets.append(lengths)
        intersected_lengths = set.intersection(*list_of_length_sets)

        # #calculate allele_weights here
        # # allele weight is   len(allele data)/ total_data_length
        # for old_allele in combination:
        #     lengths_list = {
        #         length: len(length_binders)
        #         for length, length_binders
        #         in per_allele_per_kfold_per_length_binders_train[old_allele][split_num].values()
        #     }


        for target_length in intersected_lengths:
            data_arrays_train = list()
            data_arrays_test = list()
            per_kfold_per_length_per_allele_counts_old_train[split_num][target_length] = dict()
            per_kfold_per_length_per_allele_counts_old_test[split_num][target_length] = dict()
            if split_num == 0:
                print(f"Length {target_length}:" , end=" ")
            for old_allele in combination:
                single_allele_data_array_train = per_allele_per_kfold_per_length_binders_train[old_allele][split_num][target_length]

                per_kfold_per_length_per_allele_counts_old_train[split_num][target_length][old_allele] = len(single_allele_data_array_train)
                if split_num == 0:
                    print(f"{old_allele} - {len(single_allele_data_array_train)}", end= " ")
                data_arrays_train.append(single_allele_data_array_train)
                single_allele_data_array_test = per_allele_per_kfold_per_length_binders_test[old_allele][split_num][target_length]
                per_kfold_per_length_per_allele_counts_old_test[split_num][target_length][old_allele] = len(single_allele_data_array_test)
                if split_num == 0:
                    # print(f"{old_allele} - {len(single_allele_data_array_test)}", end= " ")
                    pass
                data_arrays_test.append(single_allele_data_array_test)
            print()

            per_allele_per_kfold_per_length_binders_train_new[allele_name][split_num][target_length] = np.concatenate(data_arrays_train)
            per_allele_per_kfold_per_length_binders_test_new[allele_name][split_num][target_length] = np.concatenate(data_arrays_test)


per_allele_per_kfold_per_length_binders_train = per_allele_per_kfold_per_length_binders_train_new
per_allele_per_kfold_per_length_binders_test = per_allele_per_kfold_per_length_binders_test_new
ALLELES = NEW_ALELLES

HLA-A*01:01+HLA-A*03:01+HLA-A*02:01
Length 8: HLA-A*01:01 - 764 HLA-A*03:01 - 93 HLA-A*02:01 - 2392 
Length 9: HLA-A*01:01 - 5052 HLA-A*03:01 - 5132 HLA-A*02:01 - 24309 
Length 10: HLA-A*01:01 - 2464 HLA-A*03:01 - 2184 HLA-A*02:01 - 7129 
Length 11: HLA-A*01:01 - 1430 HLA-A*03:01 - 691 HLA-A*02:01 - 3085 
Length 12: HLA-A*01:01 - 1106 HLA-A*03:01 - 202 HLA-A*02:01 - 1077 
Length 13: HLA-A*01:01 - 842 HLA-A*03:01 - 98 HLA-A*02:01 - 726 
Length 14: HLA-A*01:01 - 568 HLA-A*03:01 - 31 HLA-A*02:01 - 556 
Length 15: HLA-A*01:01 - 464 HLA-A*03:01 - 18 HLA-A*02:01 - 555 
Length 16: HLA-A*01:01 - 345 HLA-A*03:01 - 12 HLA-A*02:01 - 400 
Length 17: HLA-A*01:01 - 209 HLA-A*03:01 - 5 HLA-A*02:01 - 252 
Length 18: HLA-A*01:01 - 128 HLA-A*03:01 - 4 HLA-A*02:01 - 162 














































## Calculate weights for data

In [30]:

def extract_count_df(per_kfold_per_length_per_allele_counts):
    result_dict = {}
    for kfold,  per_length_per_allele_counts in per_kfold_per_length_per_allele_counts.items():
        for length, per_allele_counts in per_length_per_allele_counts.items():
            for allele_name , count in  per_allele_counts.items():
                result_dict[(allele_name, kfold, length)] =  {'cnt': count }
    count_df = pd.DataFrame.from_dict(result_dict, orient="index")
    count_df.index.names = ['allele', 'split', 'length']
    # count_df = count_df.groupby(['allele', 'length'])['cnt'].mean()
    return count_df



count_df_train = extract_count_df(per_kfold_per_length_per_allele_counts_old_train)
count_df_test = extract_count_df(per_kfold_per_length_per_allele_counts_old_test)


In [31]:
count_df_train

cnt
allele      split length      
HLA-A*01:01 0     8        764
                  9       5052
                  10      2464
                  11      1430
                  12      1106
...                        ...
HLA-A*03:01 4     14        32
                  15        19
                  16        12
                  17         6
                  18         5

[165 rows x 1 columns]

# 2. Train HMMS
## 2.1 Code for HMM
### 2.1.1 Training code

In [32]:
from hmm_logic_methods import train_model, save_model, add_more_states_and_reset_transitions, init_grouped_structure_model

TARGET_LENGTHS= [9, 10]
NUM_RUNS = 2
TOTAL_SPLITS = 5

### Define weights for data

In [33]:
per_allele_per_kfold_per_length_weights = dict()


for combined_allele_name in ALLELES:
    per_allele_per_kfold_per_length_weights[combined_allele_name] = dict()
    old_alleles = combined_allele_name.split('+')
    for split_num in range(TOTAL_SPLITS):
        per_allele_per_kfold_per_length_weights[combined_allele_name][split_num] = dict()
        sub_count_df_train = count_df_train.loc[(count_df_train.index.get_level_values('split') == 0) &
                                                (count_df_train.index.get_level_values('allele').isin(old_alleles)) &
                                                (count_df_train.index.get_level_values('length').isin(TARGET_LENGTHS))]
        total_data_length = sub_count_df_train['cnt'].sum()
        print(f"Total data length for split {split_num} is {total_data_length}")
        for target_length in TARGET_LENGTHS:

            current_weights_list = list()
            weights_dict = dict()
            for old_allele_name in old_alleles:
                old_allele_cnt = count_df_train.loc[(old_allele_name, split_num, target_length)]['cnt']
                current_weights_list.extend([total_data_length/old_allele_cnt for i in range(old_allele_cnt)])
                weights_dict[old_allele_name] = total_data_length/old_allele_cnt
            per_allele_per_kfold_per_length_weights[combined_allele_name][split_num][target_length] = current_weights_list
            print(f"Weights for lentgh {target_length} are {weights_dict}")


Total data length for split 0 is 46270
Weights for lentgh 9 are {'HLA-A*01:01': 9.158749010292953, 'HLA-A*03:01': 9.01597817614965, 'HLA-A*02:01': 1.9034102595746432}
Weights for lentgh 10 are {'HLA-A*01:01': 18.77840909090909, 'HLA-A*03:01': 21.185897435897434, 'HLA-A*02:01': 6.49039135923692}
Total data length for split 1 is 46270
Weights for lentgh 9 are {'HLA-A*01:01': 9.15693647338215, 'HLA-A*03:01': 9.01597817614965, 'HLA-A*02:01': 1.9034102595746432}
Weights for lentgh 10 are {'HLA-A*01:01': 18.77079107505071, 'HLA-A*03:01': 21.185897435897434, 'HLA-A*02:01': 6.49039135923692}
Total data length for split 2 is 46270
Weights for lentgh 9 are {'HLA-A*01:01': 9.15693647338215, 'HLA-A*03:01': 9.01597817614965, 'HLA-A*02:01': 1.9033319621554916}
Weights for lentgh 10 are {'HLA-A*01:01': 18.77079107505071, 'HLA-A*03:01': 21.185897435897434, 'HLA-A*02:01': 6.489481065918653}
Total data length for split 3 is 46270
Weights for lentgh 9 are {'HLA-A*01:01': 9.15693647338215, 'HLA-A*03:01': 

## 2.2 Train
### 2.2.1 Define treshold

In [34]:
stop_threshold = 0
cutoff = 0.0001
per_symbol_portion = -50 / 9
for target_length in TARGET_LENGTHS:
    target_length_for_training = target_length
    train_size = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][target_length])
    ## Stop threshold Hardcoded here
    stop_threshold +=  abs(cutoff * train_size * per_symbol_portion  * target_length_for_training)
stop_threshold = stop_threshold / len(TARGET_LENGTHS)
stop_threshold

118.94638888888888

### 2.2.2 Prepare models

In [50]:
# Training params
algorithm = 'viterbi'

params = dict()


params['st'] = 25
params['multiple_check_input'] = False
params['alg'] = algorithm
params['min_iters'] = 5
params['stop_threshold'] = stop_threshold
params['emission_pseudocount'] = 1
params['transition_pseudocount'] = 0
params['verbose'] = True
params['lr_decay'] = 0
params['edge_inertia'] = 0.25
params['distribution_inertia'] = 0
params['groups'] = 5

assert params['st'] % params['groups'] == 0

params_for_string = {key: value for key, value in params.items() if key in ['st', 'alg', 'min_iters']}
params_string = "-".join([f"{value}-{key}" for key, value in params_for_string.items()])
lengths_string = "-".join(map(str, TARGET_LENGTHS))


In [51]:
model = init_grouped_structure_model(states_per_group=int(params['st']/params['groups']), number_of_groups=params['groups'])

grouped_model : grouped_model-start summed to 5.0, normalized to 1.0
grouped_model : s000 summed to 21.0, normalized to 1.0
grouped_model : s001 summed to 21.0, normalized to 1.0
grouped_model : s002 summed to 21.0, normalized to 1.0
grouped_model : s003 summed to 21.0, normalized to 1.0
grouped_model : s004 summed to 21.0, normalized to 1.0
grouped_model : s005 summed to 21.0, normalized to 1.0
grouped_model : s006 summed to 21.0, normalized to 1.0
grouped_model : s007 summed to 21.0, normalized to 1.0
grouped_model : s008 summed to 21.0, normalized to 1.0
grouped_model : s009 summed to 21.0, normalized to 1.0
grouped_model : s010 summed to 21.0, normalized to 1.0
grouped_model : s011 summed to 21.0, normalized to 1.0
grouped_model : s012 summed to 21.0, normalized to 1.0
grouped_model : s013 summed to 21.0, normalized to 1.0
grouped_model : s014 summed to 21.0, normalized to 1.0
grouped_model : s015 summed to 21.0, normalized to 1.0
grouped_model : s016 summed to 21.0, normalized to 

In [52]:
model.dense_transition_matrix()

array([[0.        , 0.19047619, 0.19047619, 0.19047619, 0.19047619,
        0.04761905, 0.04761905, 0.04761905, 0.04761905, 0.04761905,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        ],
       [0.19047619, 0.        , 0.19047619, 0.19047619, 0.19047619,
        0.04761905, 0.04761905, 0.04761905, 0.04761905, 0.04761905,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        ],
       [0.19047619, 0.19047619, 0.        , 0.19047619, 0.19047619,
        0.04761905, 0.04761905, 0.04761905, 0.04761905, 0.04761905,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.    

In [53]:
per_allele_per_run_per_split_prepared_models = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    per_allele_per_run_per_split_prepared_models[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = dict()
        for split_num in range(TOTAL_SPLITS):
            prepared_model  = init_grouped_structure_model(states_per_group=int(params['st']/params['groups']), number_of_groups=params['groups'])

            prepared_model.name = f'{run_num}_run-{lengths_string}-{params_string}_model-{target_allele_name}-{split_num}'
            per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = prepared_model

grouped_model : grouped_model-start summed to 5.0, normalized to 1.0
grouped_model : s000 summed to 21.0, normalized to 1.0
grouped_model : s001 summed to 21.0, normalized to 1.0
grouped_model : s002 summed to 21.0, normalized to 1.0
grouped_model : s003 summed to 21.0, normalized to 1.0
grouped_model : s004 summed to 21.0, normalized to 1.0
grouped_model : s005 summed to 21.0, normalized to 1.0
grouped_model : s006 summed to 21.0, normalized to 1.0
grouped_model : s007 summed to 21.0, normalized to 1.0
grouped_model : s008 summed to 21.0, normalized to 1.0
grouped_model : s009 summed to 21.0, normalized to 1.0
grouped_model : s010 summed to 21.0, normalized to 1.0
grouped_model : s011 summed to 21.0, normalized to 1.0
grouped_model : s012 summed to 21.0, normalized to 1.0
grouped_model : s013 summed to 21.0, normalized to 1.0
grouped_model : s014 summed to 21.0, normalized to 1.0
grouped_model : s015 summed to 21.0, normalized to 1.0
grouped_model : s016 summed to 21.0, normalized to 

In [54]:
ALLELES

['HLA-A*01:01+HLA-A*03:01+HLA-A*02:01']

In [55]:
%%time


per_name_models = dict()
per_allele_per_run_per_split_new_histories = dict()
for allele_name in ALLELES:
    per_allele_per_run_per_split_new_histories[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        print(f"run {run_num}")
        per_kfold_per_length_data = per_allele_per_kfold_per_length_binders_train[allele_name]
        per_kfold_per_length_weights = per_allele_per_kfold_per_length_weights[allele_name]


        models, histories = train_model(allele_name=allele_name,
                                        per_kfold_per_length_data=per_kfold_per_length_data,
                                        per_kfold_per_length_weights_for_data=per_kfold_per_length_weights,
                                        target_lengths=TARGET_LENGTHS,
                                        multiple_check_input=params['multiple_check_input'],
                                        algorithm=params['alg'],
                                        verbose=params['verbose'],
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test[allele_name],
                                        stop_threshold=params['stop_threshold'],
                                        min_iterations= params['min_iters'],
                                        prepared_models=per_allele_per_run_per_split_prepared_models[allele_name][run_num],
                                        emission_pseudocount=params['emission_pseudocount'],
                                        transition_pseudocount=params['transition_pseudocount'],
                                        edge_inertia=params['edge_inertia'],
                                        distribution_inertia=params['distribution_inertia'],
                                        lr_decay=params['lr_decay'])
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = models
        per_allele_per_run_per_split_new_histories[allele_name][run_num] = histories
        # per_allele_per_run_per_split_models[allele_name][run_num] = models
        for split_num, model in models.items():
            per_name_models[model.name] = model
        # save models
        for split_num, model in models.items():
            history = histories[split_num]
            save_model(TARGET_PATH_TO_RESULTS, model,history)


run 0
split 0
train size 46270
Shuffling for fairness 
Simple training: Train Length 46270, init_batch_size None
[1] Improvement: -817279.7757971976	Time (s): 7.456
[2] Improvement: 9514.473772791214	Time (s): 6.478
[3] Improvement: 3922.0942234159447	Time (s): 6.534
[4] Improvement: 3176.202089109691	Time (s): 6.592
[5] Improvement: 2077.0978081987705	Time (s): 6.513
[6] Improvement: 1600.7869823402725	Time (s): 6.401
[7] Improvement: 1814.7529752934352	Time (s): 6.629
[8] Improvement: 1145.6429410215933	Time (s): 6.587
[9] Improvement: 841.6237022727728	Time (s): 6.985
[10] Improvement: 821.6086866341066	Time (s): 7.059
[11] Improvement: 554.1674232890364	Time (s): 6.649
[12] Improvement: 459.6291962636169	Time (s): 6.895
[13] Improvement: 366.70764011424035	Time (s): 6.705
[14] Improvement: 372.00805162638426	Time (s): 7.137
[15] Improvement: 283.0954414312728	Time (s): 6.817
[16] Improvement: 291.7969936688896	Time (s): 6.487
[17] Improvement: 283.03135102335364	Time (s): 6.705
[18

In [56]:
from hmm_visualization_methods import *

In [57]:
%%time

for i, model in enumerate(per_name_models.values()):
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

0
1
Wall time: 1.39 s


In [58]:
%%time

for i, model in enumerate(per_name_models.values()):
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False)
    print(i)

0
1
Wall time: 36.6 s


In [59]:
from hmm_visualization_methods import *
per_name_pyviz = dict()
for name, model in per_name_models.items():
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=2)

Converting graph for model (0_run-9-10-25-st-viterbi-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01+HLA_A_02_01-0) (Delete zero edges)
Converting graph for model (1_run-9-10-25-st-viterbi-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01+HLA_A_02_01-0) (Delete zero edges)


## Aux viterbi

In [45]:
per_allele_per_run_per_split_new_histories = dict()

for allele_name in ALLELES:
    for run_num in range(NUM_RUNS):
        for split_num, model in per_allele_per_run_per_split_prepared_models[allele_name][run_num].items():
            prepared_model = model.copy()
            prepared_model.name = prepared_model.name + "_plus_viterbi"

            for state in prepared_model.states:
                if state is not prepared_model.start and state is not prepared_model.end:
                    state.distribution.thaw()
            per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = prepared_model

In [46]:
per_name_new_models = dict()
for allele_name in ALLELES:
    per_allele_per_run_per_split_new_histories[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        per_kfold_per_length_data = per_allele_per_kfold_per_length_binders_train[allele_name]
        per_kfold_per_length_weights = per_allele_per_kfold_per_length_weights[allele_name]
        models, histories = train_model(allele_name=allele_name,
                                        per_kfold_per_length_data=per_kfold_per_length_data,
                                        per_kfold_per_length_weights_for_data=per_kfold_per_length_weights,
                                        target_lengths=TARGET_LENGTHS,
                                        multiple_check_input=params['multiple_check_input'],
                                        algorithm='viterbi',
                                        verbose=params['verbose'],
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test[allele_name],
                                        stop_threshold=params['stop_threshold'],
                                        min_iterations= params['min_iters'],
                                        prepared_models=per_allele_per_run_per_split_prepared_models[allele_name][run_num],
                                        emission_pseudocount=params['emission_pseudocount'],
                                        transition_pseudocount= 1,  #!CHANGE for bwv!
                                        edge_inertia=params['edge_inertia'],
                                        distribution_inertia= params['distribution_inertia'],
                                        lr_decay=params['lr_decay']
                                        )
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = models
        per_allele_per_run_per_split_new_histories[allele_name][run_num] = histories
        # per_allele_per_run_per_split_models[allele_name][run_num] = models
        for split_num, model in models.items():
            per_name_new_models[model.name] = model
        # save models
        for split_num, model in models.items():
            history = histories[split_num]
            save_model(TARGET_PATH_TO_RESULTS, model,history)

split 0
train size 46270
Shuffling for fairness 
Simple training: Train Length 46270, init_batch_size None
[1] Improvement: -4856.832528775791	Time (s): 9.218
[2] Improvement: 320.7717796959914	Time (s): 6.768
[3] Improvement: 141.68962932983413	Time (s): 6.529
[4] Improvement: 77.49859326449223	Time (s): 6.65
[5] Improvement: -41.67938996292651	Time (s): 6.849
Total Training Improvement: -4358.551916448399
Total Training Time (s): 43.7944
split 0
train size 46270
Shuffling for fairness 
Simple training: Train Length 46270, init_batch_size None
[1] Improvement: -6965.776694162516	Time (s): 6.525
[2] Improvement: -2035.9136853690725	Time (s): 6.944
[3] Improvement: -836.9961116944905	Time (s): 6.605
[4] Improvement: -454.6885564646218	Time (s): 6.702
[5] Improvement: -247.52809358178638	Time (s): 6.975
Total Training Improvement: -10540.903141272487
Total Training Time (s): 40.1430


In [47]:
%%time

for i, model in enumerate(per_name_new_models.values()):
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

0
1
Wall time: 445 ms


In [48]:
%%time

for i, model in enumerate(per_name_new_models.values()):
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False)
    print(i)

0
1
Wall time: 31.2 s


In [49]:
per_name_pyviz = dict()
for name, model in per_name_new_models.items():
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=3)

Converting graph for model (0_run-9-10-25-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01+HLA_A_02_01-0_plus_viterbi) (Delete zero edges)
new starting point
Converting graph for model (1_run-9-10-25-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01+HLA_A_02_01-0_plus_viterbi) (Delete zero edges)
